## final project ##

In [ ]:
import cv2
import os
import numpy as np
from cvzone.HandTrackingModule import HandDetector

# SCREEN SIZE
screen_w, screen_h = 1920, 1080
folderpath = r"C:\Users\PRAVALLIKA\Downloads\presentation"

# CAMERA
cap = cv2.VideoCapture(0)
cam_w = int(cap.get(3))
cam_h = int(cap.get(4))

# LOAD SLIDES
pathImages = sorted(os.listdir(folderpath), key=len)

# VARIABLES
imgnumber = 0
gestureThreshold = 200
buttonPressed = False
buttonCounter = 0
buttonDelay = 20
saveCounter = 0

# ---------------- FIX: SLIDE-WISE STORAGE ----------------
annotations = {i: [] for i in range(len(pathImages))}
annotationnumber = -1
annotationstart = False
# ---------------------------------------------------------

drawCounter = 0
drawDelay = 3
drawColor = (255, 0, 0)

# POINTER
prevX, prevY = 0, 0
smoothening = 6

# CALIBRATION
minX, maxX = 0, cam_w
minY, maxY = 0, cam_h

# DRAW CONTROL
drawState = False
drawConfirm = 0
drawConfirmFrames = 4

releaseCounter = 0
releaseDelay = 6

# CAMERA PREVIEW
hs, ws = 150, 250

# HAND DETECTOR
detector = HandDetector(detectionCon=0.9, maxHands=1)

# FULLSCREEN
cv2.namedWindow("Slides", cv2.WND_PROP_FULLSCREEN)
cv2.setWindowProperty("Slides", cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)

# ZOOM VARIABLES
zoomScale = 1.0
targetZoom = 1.0
minZoom = 1.0
maxZoom = 3.0

zoomStartY = None
zoomStartScale = 1.0
isZooming = False

while True:
    success, img = cap.read()
    if not success:
        continue

    img = cv2.flip(img, 1)

    # LOAD SLIDE
    path = os.path.join(folderpath, pathImages[imgnumber])
    baseImg = cv2.imread(path)

    h, w, _ = baseImg.shape

    zoomScale += (targetZoom - zoomScale) * 0.05

    newW = int(w * zoomScale)
    newH = int(h * zoomScale)

    imgZoomed = cv2.resize(baseImg, (newW, newH))

    centerX, centerY = newW // 2, newH // 2

    startX = max(0, centerX - screen_w // 2)
    startY = max(0, centerY - screen_h // 2)

    endX = startX + screen_w
    endY = startY + screen_h

    if endX > newW:
        startX = newW - screen_w
        endX = newW
    if endY > newH:
        startY = newH - screen_h
        endY = newH

    imgcurrent = imgZoomed[startY:endY, startX:endX]
    imgcurrent = cv2.resize(imgcurrent, (screen_w, screen_h))

    # HAND DETECTION
    hands, img = detector.findHands(img)
    cv2.line(img, (0, gestureThreshold), (cam_w, gestureThreshold), (255,0,0), 5)

    if hands and not buttonPressed:
        hand = hands[0]
        fingers = detector.fingersUp(hand)

        lmList = hand['lmList']
        cx, cy = hand['center']

        # COLOR CHANGE
        if fingers == [1,1,0,0,0]:
            drawColor = (0, 0, 255)
        elif fingers == [0,0,1,1,1]:
            drawColor = (0, 255, 0)
        elif fingers == [0,0,0,1,1]:
            drawColor = (255, 0, 0)

        # ZOOM
        isZooming = False

        if fingers == [0,1,1,1,1]:
            isZooming = True

            if zoomStartY is None:
                zoomStartY = cy
                zoomStartScale = targetZoom

            deltaY = zoomStartY - cy
            targetZoom = zoomStartScale + (deltaY / 300)
            targetZoom = np.clip(targetZoom, minZoom, maxZoom)
        else:
            zoomStartY = None

        # POINTER
        x = np.clip(lmList[8][0], minX, maxX)
        y = np.clip(lmList[8][1], minY, maxY)

        xval = np.interp(x, [minX, maxX], [0, screen_w])
        yval = np.interp(y, [minY, maxY], [0, screen_h])

        currX = prevX + (xval - prevX) / smoothening
        currY = prevY + (yval - prevY) / smoothening

        currX = np.clip(currX, 0, screen_w)
        currY = np.clip(currY, 0, screen_h)

        indexFinger = int(currX), int(currY)
        prevX, prevY = currX, currY

        cv2.circle(imgcurrent, indexFinger, 12, (0, 0, 255), cv2.FILLED)

        # SLIDE CHANGE
        if cy <= gestureThreshold:

            if fingers[0] == 1 and fingers[1] == 0:
                if imgnumber > 0:
                    imgnumber -= 1
                    buttonPressed = True

            elif fingers[4] == 1 and fingers[1] == 0:
                if imgnumber < len(pathImages) - 1:
                    imgnumber += 1
                    buttonPressed = True
                    annotationnumber = -1

        else:

            if isZooming:
                drawState = False
                annotationstart = False
                drawConfirm = 0
                releaseCounter = 0

            else:
                if fingers == [0,1,0,0,0]:
                    drawConfirm += 1
                    releaseCounter = 0
                    if drawConfirm > drawConfirmFrames:
                        drawState = True
                else:
                    drawConfirm = 0
                    releaseCounter += 1
                    if releaseCounter > releaseDelay:
                        drawState = False
                        annotationstart = False
                        drawCounter = 0

                if drawState:
                    drawCounter += 1

                    if drawCounter % drawDelay == 0:

                        if not annotationstart:
                            annotationstart = True
                            annotationnumber += 1
                            annotations[imgnumber].append(([], drawColor))

                        if len(annotations[imgnumber][annotationnumber][0]) > 0:
                            prev_point = annotations[imgnumber][annotationnumber][0][-1]

                            dx = indexFinger[0] - prev_point[0]
                            dy = indexFinger[1] - prev_point[1]
                            dist = np.hypot(dx, dy)

                            if dist > 5:
                                steps = int(dist // 5)
                                for i in range(1, steps + 1):
                                    x = int(prev_point[0] + dx * i / steps)
                                    y = int(prev_point[1] + dy * i / steps)
                                    annotations[imgnumber][annotationnumber][0].append((x, y))
                        else:
                            annotations[imgnumber][annotationnumber][0].append(indexFinger)

        # ERASE
        if fingers == [0,1,1,1,0]:
            if annotations[imgnumber]:
                annotations[imgnumber].pop(-1)
                annotationnumber -= 1
                buttonPressed = True

        # ERASE ALL
        if fingers == [1,1,1,1,1]:
            annotations[imgnumber] = []
            annotationnumber = -1
            buttonPressed = True

        # SAVE
        if fingers == [1,1,1,0,0] and not buttonPressed:
            filename = rf"C:\Users\PRAVALLIKA\Downloads\{saveCounter}.png"
            cv2.imwrite(filename, imgcurrent)
            print("Saved:", filename)
            saveCounter += 1
            buttonPressed = True

    else:
        annotationstart = False

    # BUTTON RESET
    if buttonPressed:
        buttonCounter += 1
        if buttonCounter > buttonDelay:
            buttonPressed = False
            buttonCounter = 0

    # DRAW
    for points, color in annotations[imgnumber]:
        for j in range(1, len(points)):
            cv2.line(imgcurrent, points[j-1], points[j], color, 8)

    # CAMERA OVERLAY
    imgsmall = cv2.resize(img, (ws, hs))
    imgcurrent[0:hs, screen_w-ws:screen_w] = imgsmall

    cv2.imshow("Slides", imgcurrent)
    cv2.imshow("Camera", img)

    if cv2.waitKey(1) & 0xFF == ord('a'):
        break

cap.release()
cv2.destroyAllWindows()